In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, TimestampType
import pyspark.sql.functions as F


spark = (
    SparkSession.builder
    .appName("revenue")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)

schema = StructType([
    StructField("id", IntegerType(), False),
    StructField("placed_at", StringType(), True),
    StructField("status", StringType(), True),
    StructField("quantity", IntegerType(), True),
    StructField("price_each", DoubleType(), True),
    StructField("country", StringType(), True),
    StructField("book_id", IntegerType(), False),
    StructField("customer_id", IntegerType(), False),
])

df = (
    spark.read
    .option("header", True)
    .schema(schema)
    .csv("/Workspace/Users/kacperjwawrzyniak@gmail.com/Module2DE/sample_orders.csv")
)


df = (
    df
    .withColumn("placed_at", F.to_timestamp("placed_at", "yyyy-MM-dd"))
)

df = (
    df
    .withColumn("revenue", F.col("quantity") * F.col("price_each"))
)

df = (
    df
    .filter(F.col("status") == "paid")
)

df = df.withColumn(
    "country",
    F.when(F.col("country") == "Poland", "PL")
     .when(F.col("country") == "Germany", "DE")
     .when(F.col("country") == "United Kingdom", "UK")
     .when(F.col("country") == "Spain", "ES")
     .when(F.col("country") == "France", "FR")
     .when(F.col("country") == "Netherlands", "NL")
     .when(F.col("country") == "Ireland", "IE")
     .when(F.col("country") == "Sweden", "SE")
     .when(F.col("country") == "Italy", "IT")
     .when(F.col("country") == "Portugal", "PT")
     .otherwise(F.col("country"))
)


df.groupBy("country").agg(F.sum("revenue").alias("revenue"),F.count("id").alias("orders")).orderBy(F.col("revenue").desc()).show()